In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "42c76c229028988de59cbe95c701348cfecc5824"
assert (len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH"), "Pin the reviewed pushed commit before Colab validation"
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
RUN_VERSION = "v3_inverse_frequency_ce_safe_v2"
CLASS_WEIGHTING = "inverse_frequency"
CHECKPOINT_FORMAT = "frozen_backbone_head_only_v1"
SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SHARED_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-coca-runs")
V1_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier" / "v1"
V2_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier" / "v2_weighted_ce"
V2_FAILURE_RUN = V2_ROOT / "validation_runs" / "20260720T085738Z"
V3_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier" / RUN_VERSION
VALIDATION_RUNS_ROOT = V3_ROOT / "validation_runs"
VALIDATION_RECORD = V3_ROOT / "validation_record.json"
LATEST_FAILURE_RECORD = V3_ROOT / "latest_validation_failure.json"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".coca_shared_root.json"
CANDIDATE_MANIFEST = SHARED_PROJECT_DIR / "outputs" / "exploratory_balanced_ddpm" / "sqrt_balanced_seed0_v1" / "candidate_synthetic_df" / "epoch0100_seed0" / "synthetic_df.csv"
EXPECTED_CANDIDATE_SHA256 = "9ef9b44e404f74aab8211f4e7d123da3258ba8ba4e3004a4147d1761ed343b34"
V2_FAILURE_COMMIT = "0da029e8452aceaa7817e48ad6a02ec33ca03c5e"
EXPECTED_COUNTS = {"akiec": 226, "bcc": 348, "bkl": 778, "df": 585, "mel": 782, "nv": 4684, "vasc": 92}
EXPECTED_WEIGHTS = [1.3671842524688507, 0.8878840260286214, 0.39715120958606714, 0.5281771642016414, 0.3951197455984147, 0.0659657645298805, 3.3585178375865246]
EXPECTED_EQUAL_CONTRIBUTION = 308.98364105796026

# CoCa v3 inverse-frequency validation


## Phase 0: Drive, code, dependencies, and prior-run guards


In [ ]:
import hashlib, json, os, base64, shutil, subprocess, sys, time
from google.colab import drive, userdata
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project shortcut: {SHARED_PROJECT_DIR}"
assert SHARED_RUN_ROOT.is_dir(), f"missing shared run shortcut; do not create a private replacement: {SHARED_RUN_ROOT}"
assert V1_ROOT.is_dir(), f"v1 must remain present and read-only: {V1_ROOT}"
assert V2_ROOT.is_dir(), f"v2 must remain present and read-only: {V2_ROOT}"
assert SHARED_ROOT_SENTINEL.is_file(), "existing shared-root sentinel is required"
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
subprocess.run(["nvidia-smi"], check=True)
token = userdata.get("GH_TOKEN")
assert token and len(token) > 20, "Colab Secret GH_TOKEN with read access to this repo is required"
GH_TOKEN_PRESENT = True
CODE_DIR = Path("/content/ddpm-coca-v3-code")
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
basic_credential = base64.b64encode(("x-access-token:" + token).encode()).decode()
clone_env = os.environ.copy()
clone_env["GIT_CONFIG_COUNT"] = "1"
clone_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
clone_env["GIT_CONFIG_VALUE_0"] = "Authorization: Basic " + basic_credential
try:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True, env=clone_env)
finally:
    clone_env["GIT_CONFIG_VALUE_0"] = ""
    token = basic_credential = None
    del token, basic_credential, clone_env
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
remote = subprocess.check_output(["git", "-C", str(CODE_DIR), "remote", "get-url", "origin"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status, "clone must be a clean detached checkout of the pinned commit"
assert "@" not in remote and "x-access-token" not in remote, "clone URL must not embed a credential"
os.environ["HF_HOME"] = "/content/hf-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch==3.3.0", "pandas>=2.0", "pillow>=12.3.0"], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
from ddpm_derm.notebook_runtime import require_training_runtime
require_training_runtime()
from importlib.metadata import version
import torch
from ddpm_derm import coca_run
assert torch.cuda.is_available() and version("open_clip_torch") == "3.3.0"
resolved_root = coca_run.require_existing_shared_root(SHARED_RUN_ROOT)
drive_probe = coca_run.probe_shared_drive(resolved_root)
sentinel = json.loads(SHARED_ROOT_SENTINEL.read_text(encoding="utf-8"))
assert sentinel["shortcut_alias"] == "ddpm-derm-coca-runs" and sentinel["resolved_path"] == str(resolved_root)
assert sentinel["run_version"] == "v1", "v1 sentinel must not be rewritten for v3"
v1_guard_paths = [SHARED_ROOT_SENTINEL, V1_ROOT / "validation_record.json", V1_ROOT / "formal" / "_COMPLETED.json"]
v2_failure_gate_results = V2_FAILURE_RUN / "non_collapse_gate" / "results" / "coca_vit_b32"
v2_guard_paths = [v2_failure_gate_results / "results_C1_seed0.json", v2_failure_gate_results / "results_C4_seed0.json"]
guard_paths = v1_guard_paths + v2_guard_paths
assert all(path.is_file() for path in guard_paths), f"v1/v2 read-only guard files missing: {[str(p) for p in guard_paths if not p.is_file()]}"
before_guard = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
v2_failure_evidence = {}
for variant in ("C1", "C4"):
    failure_path = v2_failure_gate_results / f"results_{variant}_seed0.json"
    failure_result = json.loads(failure_path.read_text(encoding="utf-8"))
    assert failure_result["run_identity"]["run_version"] == "v2_weighted_ce"
    assert failure_result["training_objective"]["class_weighting"] == "inverse_sqrt_train_frequency"
    assert failure_result["evaluation_scope"] == "validation_only" and failure_result["test_metrics"] is None
    assert float(failure_result["best_val_df_f1"]) == 0.0
    assert failure_result["run_identity"]["git_commit"] == V2_FAILURE_COMMIT
    v2_failure_evidence[variant] = {"path": str(failure_path), "sha256": sha256(failure_path), "run_version": failure_result["run_identity"]["run_version"], "class_weighting": failure_result["training_objective"]["class_weighting"], "evaluation_scope": failure_result["evaluation_scope"], "test_metrics_null": failure_result["test_metrics"] is None, "best_val_df_f1": failure_result["best_val_df_f1"], "git_commit": failure_result["run_identity"]["git_commit"]}
assert V3_ROOT != V1_ROOT and V3_ROOT != V2_ROOT
assert V1_ROOT not in V3_ROOT.parents and V2_ROOT not in V3_ROOT.parents
if V3_ROOT.exists():
    assert not VALIDATION_RECORD.exists(), f"v3 validation_record already exists; do not re-draw validation for the same version: {VALIDATION_RECORD}"
    assert not LATEST_FAILURE_RECORD.exists(), f"v3 latest_validation_failure exists; a prior gate already failed this version: {LATEST_FAILURE_RECORD}"
coca_run.ensure_tree(SHARED_RUN_ROOT, V3_ROOT.relative_to(SHARED_RUN_ROOT))
coca_run.ensure_tree(SHARED_RUN_ROOT, VALIDATION_RUNS_ROOT.relative_to(SHARED_RUN_ROOT))

## Phase 1: Split, candidate, counts, and weights


In [ ]:
import pandas as pd
assert CANDIDATE_MANIFEST.is_file(), f"candidate manifest missing: {CANDIDATE_MANIFEST}"
candidate_hash = sha256(CANDIDATE_MANIFEST)
assert candidate_hash == EXPECTED_CANDIDATE_SHA256
candidate = pd.read_csv(CANDIDATE_MANIFEST)
assert len(candidate) == 500 and candidate["dx"].eq("df").all() and candidate["source"].eq("synthetic").all()
identity_columns = ("image_id", "lesion_id")
missing_identity_columns = [field for field in identity_columns if field not in candidate.columns]
assert not missing_identity_columns, f"candidate manifest missing identity columns: {missing_identity_columns}"
candidate_identity = {}
for field in identity_columns:
    assert candidate[field].notna().all(), f"candidate {field} contains null values"
    values = candidate[field].astype(str).str.strip()
    assert values.ne("").all(), f"candidate {field} contains empty values"
    candidate_identity[field] = set(values)
candidate_root = CANDIDATE_MANIFEST.parent
assert all((candidate_root / path).is_file() for path in candidate["image_path"])
ddpm_metadata_path = CANDIDATE_MANIFEST.parents[2] / "run_metadata.json"
ddpm_metadata = json.loads(ddpm_metadata_path.read_text(encoding="utf-8"))
assert ddpm_metadata["source_split"] == "train" and ddpm_metadata["sampler_strategy"] == "sqrt_balanced"
LOCAL_DATA_DIR = Path("/content/ham10000-data")
assert not LOCAL_DATA_DIR.exists(); shutil.copytree(SHARED_PROJECT_DIR / "data", LOCAL_DATA_DIR)
os.environ["DDPM_DERM_DATA_DIR"] = str(LOCAL_DATA_DIR)
from ddpm_derm import classifier_objective, manifests
frames = {split: manifests.load_split(split) for split in ("train", "val", "test")}
assert {key: len(value) for key, value in frames.items()} == {"train": 6995, "val": 1510, "test": 1510}
for other in ("val", "test"):
    assert not set(frames["train"]["lesion_id"]) & set(frames[other]["lesion_id"])
def assert_candidate_disjoint(split, field):
    split_values = set(frames[split][field].dropna().astype(str).str.strip())
    conflicts = sorted(candidate_identity[field] & split_values)
    assert not conflicts, f"candidate leakage: split={split} field={field} conflicts={conflicts[:10]}"
assert_candidate_disjoint("val", "image_id")
assert_candidate_disjoint("test", "image_id")
assert_candidate_disjoint("val", "lesion_id")
assert_candidate_disjoint("test", "lesion_id")
c1_full = manifests.build_classifier_frame("C1", df_target_count=585, seed=0)
c4_full = manifests.build_classifier_frame("C4", df_target_count=585, seed=0, generated_manifest=CANDIDATE_MANIFEST)
c1_objective, c1_weights = classifier_objective.build_training_objective("inverse_frequency", c1_full)
c4_objective, c4_weights = classifier_objective.build_training_objective("inverse_frequency", c4_full)
assert c1_objective == c4_objective and c1_objective["class_counts"] == EXPECTED_COUNTS
assert c1_objective["class_weighting"] == "inverse_train_frequency" and c1_objective["class_weight_formula"] == "(1/n_c)/mean_j(1/n_j)"
assert torch.allclose(torch.tensor(c1_weights), torch.tensor(EXPECTED_WEIGHTS, dtype=torch.float64), atol=5e-12, rtol=0)
assert torch.equal(torch.tensor(c1_weights), torch.tensor(c4_weights)) and abs(float(c1_weights.mean()) - 1.0) < 1e-12
contributions = [EXPECTED_COUNTS[name] * weight for name, weight in zip(classifier_objective.CLASS_ORDER, c1_weights)]
assert max(contributions) - min(contributions) < 1e-6 and abs(contributions[0] - EXPECTED_EQUAL_CONTRIBUTION) < 1e-6
fixed_split_identity = sha256(LOCAL_DATA_DIR / "manifests" / "train.csv")
print(json.dumps(c1_objective, indent=2))

## Phases 2-3: CoCa forward, freeze, optimizer, and criterion


In [ ]:
import open_clip
from PIL import Image
from ddpm_derm.model import build_model, model_identity, parameter_counts
from ddpm_derm.train_classifier import build_criterion, build_optimizer
assert ("coca_ViT-B-32", "laion2b_s13b_b90k") in set(map(tuple, open_clip.list_pretrained()))
model = build_model(arch="coca_vit_b32", freeze_backbone=True, coca_pretrained="laion2b_s13b_b90k").cuda()
sample_paths = [LOCAL_DATA_DIR / path for path in frames["train"]["image_path"].head(2)]
batch = torch.stack([model.eval_preprocess(Image.open(path).convert("RGB")) for path in sample_paths]).cuda()
model.train(); logits = model(batch)
assert tuple(logits.shape) == (2, 7) and model.input_resolution == (224, 224)
assert not model.encoder.training and all(not p.requires_grad for p in model.encoder.parameters())
assert all(p.requires_grad for p in model.head.parameters())
total_params, trainable_params = parameter_counts(model)
assert trainable_params == 3591, f"trainable head params expected 3591, got {trainable_params}"
assert total_params == 253563656, f"total params expected 253563656, got {total_params}"
model.zero_grad(set_to_none=True)
model(batch).sum().backward()
assert all(p.grad is None for p in model.encoder.parameters()), "frozen encoder must build no gradient graph"
assert any(p.grad is not None for p in model.head.parameters()), "head must receive gradients"
model.zero_grad(set_to_none=True)
optimizer = build_optimizer(model, 3e-4, 1e-4)
assert {id(p) for group in optimizer.param_groups for p in group["params"]} == {id(p) for p in model.head.parameters()}
criterion = build_criterion("inverse_frequency", c1_weights, torch.device("cuda"))
assert criterion.weight is not None and tuple(criterion.weight.shape) == (7,)
assert criterion.weight.dtype == torch.float32 and criterion.weight.device.type == "cuda"
assert torch.equal(criterion.weight.cpu(), torch.tensor(c1_weights, dtype=torch.float32))
model_details = model_identity(model, "coca_vit_b32", 128)
assert model_details["open_clip_torch_version"] == "3.3.0"
del model, optimizer, criterion, batch, logits; torch.cuda.empty_cache()

## Phase 4: One-epoch checkpoint, resume, and mismatch smoke test


In [ ]:
from ddpm_derm.checkpoint import load_checkpoint

import copy
from datetime import datetime, timezone
from ddpm_derm import classifier_run
validation_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
VALIDATION_DIR = coca_run.ensure_tree(SHARED_RUN_ROOT, (VALIDATION_RUNS_ROOT / validation_id).relative_to(SHARED_RUN_ROOT))
SMOKE_ROOT = coca_run.ensure_tree(SHARED_RUN_ROOT, (VALIDATION_DIR / "smoke").relative_to(SHARED_RUN_ROOT))
formal_output_identity = f"{sentinel['shared_root_uuid']}:sqrt_balanced_seed0_v1:coca_classifier:{RUN_VERSION}:formal"
env = os.environ.copy(); env["PYTHONPATH"] = str(CODE_DIR / "src"); env["PYTHONUNBUFFERED"] = "1"
def run_stream(command, expect_success=True):
    started = time.monotonic(); process = subprocess.Popen(command, cwd=CODE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout: lines.append(line); print(line, end="", flush=True)
    code = process.wait()
    if expect_success and code: raise subprocess.CalledProcessError(code, command)
    if not expect_success and not code: raise AssertionError("command unexpectedly succeeded")
    return time.monotonic() - started, "".join(lines)
common = [sys.executable, "-u", "-m", "ddpm_derm.train_classifier", "--arch", "coca_vit_b32", "--freeze-backbone", "--coca-pretrained", "laion2b_s13b_b90k", "--seed", "0", "--epochs", "1", "--batch-size", "8", "--lr", "3e-4", "--weight-decay", "1e-4", "--df-target-count", "585", "--limit", "64", "--num-workers", "2", "--class-weighting", "inverse_frequency", "--evaluation-scope", "validation_only", "--output-dir", str(SMOKE_ROOT), "--run-version", RUN_VERSION, "--shared-root-uuid", sentinel["shared_root_uuid"], "--formal-output-identity", formal_output_identity, "--fixed-split-identity", fixed_split_identity, "--candidate-sha256", candidate_hash]
smoke_commands = {}; smoke_checks = {}
for variant in ("C1", "C4"):
    command = common + ["--variant", variant, "--run-label", f"coca_v3_smoke_{variant}"]
    if variant == "C4": command += ["--generated-manifest", str(CANDIDATE_MANIFEST)]
    smoke_commands[variant] = command
    _, output = run_stream(command)
    assert "checkpoint_saved=last.pt" in output and "[test]" not in output
    checkpoint_dir = SMOKE_ROOT / "checkpoints" / "coca_vit_b32" / f"{variant}_seed0"
    result_path = SMOKE_ROOT / "results" / "coca_vit_b32" / f"results_{variant}_seed0.json"
    last_path = checkpoint_dir / "last.pt"; saved = load_checkpoint(last_path, map_location="cpu")
    result = json.loads(result_path.read_text(encoding="utf-8"))
    assert saved["checkpoint_format"] == CHECKPOINT_FORMAT and "head_state_dict" in saved and "model_state_dict" not in saved
    assert result["evaluation_scope"] == "validation_only" and result["test_metrics"] is None
    assert saved["run_identity"]["training_objective"] == c1_objective
    before = (sha256(last_path), last_path.stat().st_mtime_ns)
    _, resumed = run_stream(command + ["--resume"]); assert "[resume]" in resumed and "[skip]" in resumed
    for wrong_mode in ("none", "inverse_sqrt"):
        mismatch = command.copy(); mismatch[mismatch.index("--class-weighting") + 1] = wrong_mode
        run_stream(mismatch + ["--resume"], expect_success=False)
    if variant == "C1":
        mismatch = command.copy(); mismatch[mismatch.index("585")] = "586"
        run_stream(mismatch + ["--resume"], expect_success=False)
    changed = copy.deepcopy(saved["run_identity"]); changed["evaluation_scope"] = "full"
    try: classifier_run.require_matching_resume_identity(saved["run_identity"], changed); raise AssertionError("evaluation scope")
    except ValueError: pass
    changed = copy.deepcopy(saved["run_identity"]); changed["run_version"] = "v2_weighted_ce"
    try: classifier_run.require_matching_resume_identity(saved["run_identity"], changed); raise AssertionError("run version")
    except ValueError: pass
    changed = copy.deepcopy(saved["run_identity"]); changed["model_identity"]["pretrained_tag"] = "other"
    try: classifier_run.require_matching_resume_identity(saved["run_identity"], changed); raise AssertionError("model identity")
    except ValueError: pass
    changed = copy.deepcopy(saved["run_identity"]); changed["training_objective"]["class_weight_formula"] = "(1/sqrt(n_c))/mean_j(1/sqrt(n_j))"
    try: classifier_run.require_matching_resume_identity(saved["run_identity"], changed); raise AssertionError("formula")
    except ValueError: pass
    changed = copy.deepcopy(saved["run_identity"]); changed["training_objective"]["class_counts"] = {**EXPECTED_COUNTS, "df": 584}
    try: classifier_run.require_matching_resume_identity(saved["run_identity"], changed); raise AssertionError("counts")
    except ValueError: pass
    changed = copy.deepcopy(saved["run_identity"]); changed["training_objective"]["class_weights"][0] += 0.1
    try: classifier_run.require_matching_resume_identity(saved["run_identity"], changed); raise AssertionError("weights")
    except ValueError: pass
    assert (sha256(last_path), last_path.stat().st_mtime_ns) == before
    smoke_checks[variant] = {"resume": True, "mismatches_rejected_without_mutation": True}
child_code = "import subprocess,sys; subprocess.run(sys.argv[1:], check=True)"
subprocess.run([sys.executable, "-c", child_code] + smoke_commands["C1"] + ["--resume"], cwd=CODE_DIR, env=env, check=True)
smoke_checks["child_process_drive_only_restore"] = True

## Phase 5: Five-epoch non-collapse gate


In [ ]:
GATE_ROOT = coca_run.ensure_tree(SHARED_RUN_ROOT, (VALIDATION_DIR / "non_collapse_gate").relative_to(SHARED_RUN_ROOT))
gate_common = [sys.executable, "-u", "-m", "ddpm_derm.train_classifier", "--arch", "coca_vit_b32", "--freeze-backbone", "--coca-pretrained", "laion2b_s13b_b90k", "--seed", "0", "--epochs", "5", "--batch-size", "32", "--lr", "3e-4", "--weight-decay", "1e-4", "--df-target-count", "585", "--num-workers", "2", "--class-weighting", "inverse_frequency", "--evaluation-scope", "validation_only", "--output-dir", str(GATE_ROOT), "--run-version", RUN_VERSION, "--shared-root-uuid", sentinel["shared_root_uuid"], "--formal-output-identity", formal_output_identity, "--fixed-split-identity", fixed_split_identity, "--candidate-sha256", candidate_hash]
gate_metrics = {}; gate_times = {}; gate_failures = []
for variant in ("C1", "C4"):
    command = gate_common + ["--variant", variant, "--run-label", f"coca_v3_gate_{variant}"]
    if variant == "C4": command += ["--generated-manifest", str(CANDIDATE_MANIFEST)]
    gate_times[variant], output = run_stream(command)
    result_path = GATE_ROOT / "results" / "coca_vit_b32" / f"results_{variant}_seed0.json"
    checkpoint_dir = GATE_ROOT / "checkpoints" / "coca_vit_b32" / f"{variant}_seed0"
    result = json.loads(result_path.read_text(encoding="utf-8"))
    metrics = result["validation_metrics"]; matrix = torch.tensor(metrics["confusion_matrix"]); predicted = matrix.sum(dim=0)
    prediction_counts = {name: int(predicted[index]) for index, name in enumerate(classifier_objective.CLASS_ORDER)}
    checks = {"best_validation_df_f1_positive": result["best_val_df_f1"] > 0, "predicted_df_positive": prediction_counts["df"] > 0, "not_all_nv": prediction_counts["nv"] < int(predicted.sum()), "at_least_two_predicted_classes": int((predicted > 0).sum()) >= 2, "identity_complete": result["run_identity"]["training_objective"] == c1_objective, "test_metrics_null": result["test_metrics"] is None, "no_test_log": "[test]" not in output}
    if not all(checks.values()): gate_failures.append({"variant": variant, "checks": checks})
    assert (checkpoint_dir / "best.pt").is_file() and (checkpoint_dir / "last.pt").is_file()
    best_epoch = max(result["history"], key=lambda item: item["val_df_f1"])["epoch"] if result["history"] else None
    gate_metrics[variant] = {"best_validation_df_f1": result["best_val_df_f1"], "best_epoch": best_epoch, "history": result["history"], "prediction_counts": prediction_counts, "checks": checks, "elapsed_seconds": gate_times[variant]}
if gate_failures:
    failure_record = {"validation_status": "VALIDATION FAILED", "formal_training_started": False, "class_weighting": "inverse_train_frequency", "git_commit": commit, "run_version": RUN_VERSION, "candidate_manifest_sha256": candidate_hash, "fixed_split_identity": fixed_split_identity, "shared_root_uuid": sentinel["shared_root_uuid"], "class_weight_formula": c1_objective["class_weight_formula"], "class_counts": c1_objective["class_counts"], "class_weights": c1_objective["class_weights"], "model_identity": model_details, "gate_failures": gate_failures, "non_collapse_gate": gate_metrics, "resume_mismatch_checks": smoke_checks, "drive_probes": drive_probe, "validation_artifact_directory": str(VALIDATION_DIR), "test_metrics": None}
    coca_run.write_json_atomic(VALIDATION_DIR / "validation_failure.json", failure_record)
    coca_run.write_json_atomic(LATEST_FAILURE_RECORD, failure_record)
    assert not VALIDATION_RECORD.exists(), "a failed gate must never leave a success validation_record"
    print("VALIDATION FAILED")
    print("formal_training_started=false")
    print("class_weighting=inverse_train_frequency")
    raise RuntimeError(gate_failures)

## Phase 6: Validation record and stop before formal training


In [ ]:
guard_after = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
assert guard_after == before_guard, "v1/v2 read-only guard files changed"
equal_contribution = [EXPECTED_COUNTS[name] * weight for name, weight in zip(classifier_objective.CLASS_ORDER, c1_objective["class_weights"])]
estimated_seconds = float(sum(gate_times.values()) / 2 / 5 * 20 * 6)
record = {"validation_status": "VALIDATION PASSED", "formal_training_started": False, "class_weighting": "inverse_train_frequency", "git_commit": commit, "run_version": RUN_VERSION, "candidate_manifest_sha256": candidate_hash, "fixed_split_identity": fixed_split_identity, "shared_root_uuid": sentinel["shared_root_uuid"], "shared_root_identity": sentinel, "formal_output_identity": formal_output_identity, "training_objective": c1_objective, "class_counts": c1_objective["class_counts"], "class_weight_formula": c1_objective["class_weight_formula"], "class_weights": c1_objective["class_weights"], "equal_class_contribution": equal_contribution, "equal_class_contribution_spread": max(equal_contribution) - min(equal_contribution), "model_identity": model_details, "dependency_versions": {"open_clip_torch": version("open_clip_torch"), "torch": torch.__version__}, "checkpoint_format": CHECKPOINT_FORMAT, "evaluation_scope": "validation_only", "non_collapse_gate": gate_metrics, "gate_best_epochs": {variant: gate_metrics[variant]["best_epoch"] for variant in ("C1", "C4")}, "checkpoint_resume_mismatch_checks": smoke_checks, "drive_probes": drive_probe, "validation_artifact_directory": str(VALIDATION_DIR), "estimated_formal_six_run_seconds": estimated_seconds, "estimate_basis": "two five-epoch full-data validation-only gates scaled to six 20-epoch runs", "v1_v2_artifacts_unchanged": True, "v2_failure_evidence": v2_failure_evidence, "gh_token_present": GH_TOKEN_PRESENT}
coca_run.write_json_atomic(VALIDATION_DIR / "validation_record.json", record)
coca_run.write_json_atomic(VALIDATION_RECORD, record)
assert not (V3_ROOT / "formal" / "_RUNNING.json").exists()
print(json.dumps(record, indent=2))
print("VALIDATION PASSED")
print("formal_training_started=false")
print("class_weighting=inverse_train_frequency")